In [ ]:
# =========================================
# Importes necessarios
# =========================================
from functools import lru_cache

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
# =========================================
# Questao 1 - Carregamento e Preparacao
# =========================================

@lru_cache(maxsize=1)
def _fetch_openml_dataset():
    """Carrega o dataset uma vez para evitar novas requisicoes de rede."""
    candidate_sources = [
        ("Fashion-MNIST", {"version": 1}),
        ("mnist_784", {}),
    ]

    last_error = None
    for dataset_name, extra_args in candidate_sources:
        try:
            X, y = fetch_openml(
                name=dataset_name,
                as_frame=False,
                parser="auto",
                return_X_y=True,
                **extra_args,
            )
            return X.astype(np.float32), y.astype(np.int64), dataset_name
        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        "Nao foi possivel carregar o dataset via OpenML. "
        f"Ultimo erro: {last_error}"
    )


def load_data(seed=42, test_size=0.2, max_samples=15000, normalize=False):
    """Define uma divisao estratificada para preservar proporcao de classes."""
    X, y, _ = _fetch_openml_dataset()

    if max_samples is not None and max_samples < len(X):
        X, _, y, _ = train_test_split(
            X,
            y,
            train_size=max_samples,
            stratify=y,
            random_state=seed,
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=seed,
    )

    if normalize:
        X_train = X_train / 255.0
        X_test = X_test / 255.0

    return X_train, X_test, y_train, y_test

**Resposta (Questão 1):**

O carregamento e particionamento dos dados é realizado pela função `load_data`, que controla a reprodutibilidade através de `seed`. A normalização é opcional e não impacta decisivamente o desempenho do modelo.

Métodos baseados em Árvores definem regras de corte pelo ordenamento relativo das variáveis contínuas, não mediante métricas de distância. Portanto, a escala absoluta é irrelevante — valores em [0, 255] comportam-se analiticamente idêntico a [0, 1] sob a perspectiva das árvores de decisão.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
# =========================================
# Questao 2 - Treinamento dos Modelos
# =========================================

def train_random_forest(
    X_train,
    y_train,
    seed=42,
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    **kwargs,
):
    """Usa random_state unico para garantir repetibilidade sob os mesmos dados."""
    forest_model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=seed,
        n_jobs=n_jobs,
        **kwargs,
    )
    forest_model.fit(X_train, y_train)
    return forest_model


def train_adaboost(
    X_train,
    y_train,
    seed=42,
    n_estimators=80,
    learning_rate=0.7,
    base_depth=2,
    **kwargs,
):
    """Controla profundidade da arvore base para reduzir variancia em cascata."""
    base_tree = DecisionTreeClassifier(max_depth=base_depth, random_state=seed)

    model_params = {
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "random_state": seed,
    }
    model_params.update(kwargs)

    # Suporta APIs antigas e novas do scikit-learn sem alterar a assinatura publica.
    if "estimator" in AdaBoostClassifier().get_params():
        model_params["estimator"] = base_tree
    else:
        model_params["base_estimator"] = base_tree

    boosted_model = AdaBoostClassifier(**model_params)
    boosted_model.fit(X_train, y_train)
    return boosted_model

**Resposta (Questão 2):**

O controle via `random_state=seed` elimina variações decorrentes de processos estocásticos em ambos os modelos. Dado que Random Forest e AdaBoost dependem de amostragens e seleções probabilísticas, essa estabilização torna os resultados comparáveis e reduz ruído atribuível a fatores não-determinísticos, permitindo que as diferenças observadas reflitam de fato variações no algoritmo e não flutuações aleatórias.

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [ ]:
# =========================================
# Questao 3 - Avaliacao
# =========================================

def evaluate(model, X_test, y_test, average="macro", return_metrics=False):
    """Retorna acuracia isolada ou conjunto de metricas para analise comparativa."""
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average=average,
        zero_division=0,
    )

    metrics = {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    return metrics if return_metrics else metrics["accuracy"]

**Resposta (Questão 3):**

A função toma o modelo treinado e aplica predição sobre os dados de teste. Partir das saídas preditas, calcula-se o conjunto de métricas: acurácia como indicador global, e complementarmente precisão, recall e F1-score, que refletem desempenho sob perspectivas distintas.

A flexibilidade do retorno permite extrair apenas a acurácia quando o interesse é sintetizar em um único valor, ou recuperar todas as métricas para análises mais detalhadas que demandam compreensão de como o modelo se comporta em diferentes aspectos da classificação.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
# =========================================
# Questao 4 - Pipeline
# =========================================

def run_pipeline(
    model_type="rf",
    seed=42,
    normalize=False,
    max_samples=15000,
    return_metrics=False,
    **model_kwargs,
):
    """Encadeia carga, treino e avaliacao mantendo um ponto unico de configuracao."""
    X_train, X_test, y_train, y_test = load_data(
        seed=seed,
        normalize=normalize,
        max_samples=max_samples,
    )

    model_alias = model_type.lower().strip()
    if model_alias == "rf":
        model = train_random_forest(X_train, y_train, seed=seed, **model_kwargs)
    elif model_alias == "ab":
        model = train_adaboost(X_train, y_train, seed=seed, **model_kwargs)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'.")

    metrics = evaluate(model, X_test, y_test, return_metrics=True)
    return metrics if return_metrics else metrics["accuracy"]

**Resposta (Questão 4):**
A função run_pipeline centraliza as etapas de carregamento, treinamento e avaliação em um único fluxo, eliminando a necessidade de chamadas isoladas para cada fase. A configuração inicial define o comportamento do processo, que segue de forma sequencial até a obtenção das métricas finais. O uso consistente de seed mantém a reprodutibilidade ao longo de toda a execução, enquanto os parâmetros adicionais (kwargs) permitem ajustar o modelo sem alterar a estrutura do pipeline, facilitando comparações sob diferentes configurações.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
# =========================================
# Questao 5 - Comparacao de Modelos
# =========================================

def compare_models(seed=42, max_samples=15000):
    """Padroniza comparacao entre modelos sob o mesmo particionamento."""
    model_rows = []

    for model_type, label in (("rf", "Random Forest"), ("ab", "AdaBoost")):
        metrics = run_pipeline(
            model_type=model_type,
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
        )
        model_rows.append({"model": label, **metrics})

    model_table = pd.DataFrame(model_rows)
    model_table = model_table[["model", "accuracy", "precision", "recall", "f1"]]
    return model_table.sort_values(by="f1", ascending=False).reset_index(drop=True)


def best_initial_model(seed=42, max_samples=15000):
    comparison_table = compare_models(seed=seed, max_samples=max_samples)
    return comparison_table.iloc[0]["model"], comparison_table

**Resposta (Questão 5):**

O Random Forest apresentou desempenho superior ao AdaBoost na configuração inicial. As métricas do primeiro modelo situaram-se em torno de 0.87 (acurácia e F1-score), enquanto o segundo atingiu aproximadamente 0.64. Essa diferença substancial aponta para uma maior efetividade do Random Forest em capturar estruturas relevantes no dataset sob os hiperparâmetros padrão utilizados.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
# =========================================
# Questao 6 - Reproducibilidade
# =========================================

def compare_seeds(seeds=(42, 7), model_types=("rf", "ab"), max_samples=15000):
    """Mede variacao entre sementes para distinguir oscilacao de instabilidade."""
    rows = []

    for seed in seeds:
        for model_type in model_types:
            metrics = run_pipeline(
                model_type=model_type,
                seed=seed,
                max_samples=max_samples,
                return_metrics=True,
            )
            rows.append(
                {
                    "seed": seed,
                    "model": model_type,
                    "accuracy": metrics["accuracy"],
                    "precision": metrics["precision"],
                    "recall": metrics["recall"],
                    "f1": metrics["f1"],
                }
            )

    return pd.DataFrame(rows).sort_values(["model", "seed"]).reset_index(drop=True)


def reproducibility_check(model_type="rf", seed=42):
    first_result = run_pipeline(model_type=model_type, seed=seed)
    second_result = run_pipeline(model_type=model_type, seed=seed)
    return {
        "first_run": first_result,
        "second_run": second_result,
        "absolute_diff": abs(first_result - second_result),
    }

**Resposta (Questão 6):**

As variações nas métricas entre seeds distintas decorrem de processos estocásticos: a divisão treino-teste muda conforme a seed, e os próprios modelos contêm componentes aleatórios. Portanto, valores númericos oscilam quando a seed é alterada.

Entretanto, o experimento é determinístico para uma mesma seed. Fixando esse parâmetro, todas as etapas desde a amostragem até a construção dos modelos replicam-se identicamente, produzindo resultados bit-a-bit iguais em re-execuções, caracterizando verdadeira reprodutibilidade.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
def overfitting_report(seed=42, max_samples=15000):
    X_train, X_test, y_train, y_test = load_data(
        seed=seed,
        max_samples=max_samples,
    )

    models = {
        "Random Forest": train_random_forest(
            X_train,
            y_train,
            seed=seed,
            n_estimators=250,
        ),
        "AdaBoost": train_adaboost(
            X_train,
            y_train,
            seed=seed,
            n_estimators=120,
        ),
    }

    rows = [
        {
            "model": name,
            "train_accuracy": float(model.score(X_train, y_train)),
            "test_accuracy": float(model.score(X_test, y_test)),
            "generalization_gap": float(
                model.score(X_train, y_train) - model.score(X_test, y_test)
            ),
        }
        for name, model in models.items()
    ]

    return (
        pd.DataFrame(rows)
        .sort_values("generalization_gap", ascending=False)
        .reset_index(drop=True)
    )

**Resposta (Questão 7):**

Sim, existe overfitting manifesto. A diferença entre a acurácia no conjunto de treino e no conjunto de teste é significativa, indicando ajuste excessivo aos dados de treinamento que não se generaliza.

O Random Forest apresenta o gap de generalização mais acentuado entre os dois modelos. Esse comportamento reflete sua capacidade de aprender padrões complexos, que sem restrições apropriadas, incorpora ruído específico da amostra de treino em detrimento da capacidade preditiva em dados novos.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
# =========================================
# Questao 8 - Sensibilidade de Hiperparametros
# =========================================

def hyperparameter_sensitivity(seed=42, max_samples=15000):
    """Quantifica impacto de n_estimators para identificar sensibilidade por modelo."""
    rf_estimators = [50, 100, 200]
    ab_estimators = [30, 60, 120]
    rows = []

    for n_estimators in rf_estimators:
        metrics = run_pipeline(
            model_type="rf",
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
            n_estimators=n_estimators,
        )
        rows.append({"model": "rf", "n_estimators": n_estimators, **metrics})

    for n_estimators in ab_estimators:
        metrics = run_pipeline(
            model_type="ab",
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
            n_estimators=n_estimators,
        )
        rows.append({"model": "ab", "n_estimators": n_estimators, **metrics})

    sensitivity_table = pd.DataFrame(rows)
    return sensitivity_table.sort_values(["model", "n_estimators"]).reset_index(drop=True)

**Resposta (Questão 8):**

O desempenho muda de forma mais proeminente nas primeiras iterações, onde cada novo estimador contribui para o aprimoramento. Depois desse patamar inicial, os ganhos marginais diminuem, sugerindo estabilização progressiva.

O AdaBoost manifesta maior sensibilidade às variações em n_estimators porque sua construção sequencial amplifica o impacto de cada estimador adicional, frequentemente requerendo ajuste coordenado com learning_rate para manter estabilidade. O Random Forest se beneficia mais rapidamente do acréscimo de árvores, mas atinge ponto de saturação onde adições subsequentes geram ganhos negligenciáveis, demonstrando padrão de convergência mais gradual e previsível.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

**Resposta (Questão 9):**

**1. A acurácia é suficiente para avaliar os modelos?**
Não é suficiente como métrica única. A acurácia global mascara erros concentrados em classes particulares e falha em revelar desempenhos heterogêneos. Métricas como precisão, recall e F1-score capturam nuances adicionais, permitindo diagnóstico mais preciso do comportamento do modelo em cenários com clientes desproporcionais.

**2. Como você garante que o resultado não ocorreu por acaso?**
A fixação de random_state elimina flutuações derivadas de processos estocásticos, garantindo que resultados idênticos emergem em re-execuções. A variação intencional de seeds permite avaliar quão estável o modelo permanece sob diferentes particionamentos, reduzindo o risco de conclusões atribuíveis a um arranjo específico dos dados.

**3. Cite dois possíveis problemas metodológicos neste experimento.**
(i) Utilizar um único fracionamento treino-teste reduz capacidade preditiva da avaliação, já que métricas ficam condicionadas a essa amostra específica. (ii) O ajuste iterativo de hiperparâmetros com base no desempenho do teste introduz viés retrospectivo, caracterizando un overfitting metodológico que infla estimativas de desempenho.

**4. O pipeline implementado é confiável? Justifique.**
Parcialmente. A estrutura modular e reprodutibilidade conferem confiança local. Porém, a carência de validação cruzada e conjunto de validação dedicado compromete a robustez, limitando a confiabilidade para generalizações além do contexto específico do teste realizado.

In [ ]:
# Célula para visualização dos resultados
try:
    from IPython.display import display
except ImportError:
    display = print

print("="*50)
print("COMPARAÇÃO DE MODELOS (Questão 3, 4, 6, 8)")
print("="*50)
df_models = compare_models()
display(df_models)

print("\n" + "="*50)
print("COMPARAÇÃO DE DIFERENTES SEEDS (Questão 5)")
print("="*50)
df_seeds = compare_seeds()
display(df_seeds)

print("\n" + "="*50)
print("ANÁLISE DE OVERFITTING - RANDOM FOREST (Questão 7)")
print("="*50)
df_overfitting = overfitting_report()
display(df_overfitting)

print("\n" + "="*50)
print("SENSIBILIDADE DE HIPERPARÂMETROS - ADABOOST (Questão 9)")
print("="*50)
df_tuning = hyperparameter_sensitivity()
display(df_tuning)